In [272]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [273]:
from Base_Modules.Environments import Prison
from Prison_Strategies.Basic_Strategies import *
from Prison_Strategies.Reinforcement_Learning_Strategies import *
from Base_Modules.Game_Master import Game_Master
from Base_Modules.Action import Action_History, Prison_Actions, Duel_Matrix
import pandas as pd
from collections import defaultdict


In [274]:
prison = Prison()
actions = prison.Get_Actions()


In [275]:

def Get_Enemy_Strategies():
    return [
        Random_Strategy(actions=actions),
        Random_Strategy(actions=actions, p_coop=0.1),
        Always_Betray(actions=actions),
        Always_Cooperate(actions=actions),
        Patient_Unforgiving(actions=actions),
        Copycat(actions=actions),
        Periodic(actions=actions, period=1),
        Forgiving(actions=actions),
    ]


def Enumerate_Strategies(strategies_list : list[Strategy]) -> dict[int, Strategy]:
    strategies = {}
    for (i, s) in enumerate(strategies_list):
        strategies[i] = s
        s.Set_ID(i)
    return strategies

In [276]:
def Raport(lr, gamma, state_size, eps_decay, eps_eval, iteration, possibilites, average_q_score):
    print(f"Progress={(iteration/possibilites):.2f}, Score={average_q_score:.2f} lr={lr}, gamma={gamma}, state_size={state_size}, eps_decay={eps_decay}, eps_eval={eps_eval}")

def Normal_Tournament(gm : Game_Master, number_of_games, total_games_explicit) -> tuple[Duel_Matrix, object]:
    return gm.Tournament(number_of_games, Game_Master.Game_Type.All_Vs_All, total_games_explicit=total_games_explicit)

def RL_Training(gm : Game_Master, number_of_games, number_of_RL_epochs, total_games_explicit) -> Game_Master:
    for i in range(number_of_RL_epochs):
        gm.Reset()
        gm.Tournament(number_of_games, Game_Master.Game_Type.All_Vs_All, total_games_explicit=total_games_explicit)
        if (i+1)%1000 == 0:
            print(i)
    return gm

In [277]:

lrs = [0.01, 0.001]
gammas = [0.99, 1.01]
state_sizes = [1, 2, 3]
eps_decays = [0.99, 0.999, 0.9999]
eps_evals = [0.0, 0.0001]

possibilities = len(lrs) * len(gammas) * len(state_sizes) * len(eps_decays) * len(eps_evals)

number_of_games_per_epoch = 80
number_of_RL_epochs = 1500
total_games_explicit = True
max_action_memory = -1

eps_init = 1.0
eps_min = 0.05

In [278]:
final_scores = {}
i=0
best_score = -float("inf")
foldername = "Q_Learning_Models/Training_Ground_Models/"
best_q_save_path = foldername + "best_q_learning_so_far.pkl"


In [279]:
for lr in lrs:
    for gamma in gammas:
        for state_size in state_sizes:
            for eps_decay in eps_decays:
                for eps_eval in eps_evals:
                    strategies_list = Get_Enemy_Strategies()
                    q_learning = Q_Learning(actions=actions,
                                            eps=eps_init,
                                            eps_decay=eps_decay,
                                            eps_min=eps_min,
                                            gamma=gamma,
                                            lr=lr,
                                            state_size=state_size,
                                            register_enemy_ID=False,
                                            eps_eval=eps_eval)
                    strategies_list.append(q_learning)
                    number_of_strategies = len(strategies_list)
                    strategies = Enumerate_Strategies(strategies_list)
                    gm = Game_Master(environment=prison, strategies=strategies, duel_size=2, max_action_memory=max_action_memory)
                    q_learning.Train()
                    RL_Training(gm=gm, number_of_games=number_of_games_per_epoch, number_of_RL_epochs=number_of_RL_epochs, total_games_explicit=total_games_explicit)
                    q_learning.Eval()
                    gm.Reset()
                    duel_matrix, rewards = Normal_Tournament(gm=gm, number_of_games=number_of_games_per_epoch, total_games_explicit=total_games_explicit)
                    rewards.Sort_Total_Rewards()
                    average_q_score = rewards.Get_Total_Rewards()[q_learning.Get_ID()]/(number_of_games_per_epoch*number_of_strategies)
                    final_scores[i] = {
                        "average_score" : average_q_score,
                        "lr" : lr,
                        "gamma" : gamma,
                        "state_size" : state_size,
                        "eps_decay" : eps_decay,
                        "eps_eval" : eps_eval,
                        "object" : q_learning
                    }
                    if average_q_score > best_score:
                        best_score = average_q_score
                        print("Achieved best score.")
                        q_learning.Save_Q_Table(filename=best_q_save_path)

                    Raport(lr, gamma, state_size, eps_decay, eps_eval, i, possibilities, average_q_score)
                    i+=1

final_scores = dict(sorted(
    final_scores.items(),
    key=lambda x: x[1]["average_score"],
    reverse=True
))

999
Achieved best score.
Progress=0.00, Score=1.81 lr=0.01, gamma=0.99, state_size=1, eps_decay=0.99, eps_eval=0.0
999
Achieved best score.
Progress=0.01, Score=1.90 lr=0.01, gamma=0.99, state_size=1, eps_decay=0.99, eps_eval=0.0001
999
Progress=0.03, Score=1.88 lr=0.01, gamma=0.99, state_size=1, eps_decay=0.999, eps_eval=0.0
999
Progress=0.04, Score=1.36 lr=0.01, gamma=0.99, state_size=1, eps_decay=0.999, eps_eval=0.0001
999
Progress=0.06, Score=1.84 lr=0.01, gamma=0.99, state_size=1, eps_decay=0.9999, eps_eval=0.0
999
Progress=0.07, Score=1.89 lr=0.01, gamma=0.99, state_size=1, eps_decay=0.9999, eps_eval=0.0001
999
Progress=0.08, Score=1.64 lr=0.01, gamma=0.99, state_size=2, eps_decay=0.99, eps_eval=0.0
999
Progress=0.10, Score=1.56 lr=0.01, gamma=0.99, state_size=2, eps_decay=0.99, eps_eval=0.0001
999
Progress=0.11, Score=1.72 lr=0.01, gamma=0.99, state_size=2, eps_decay=0.999, eps_eval=0.0
999
Progress=0.12, Score=1.71 lr=0.01, gamma=0.99, state_size=2, eps_decay=0.999, eps_eval=0.

In [280]:
df = pd.DataFrame.from_dict(final_scores, orient="index")
df

,average_score,lr,gamma,state_size,eps_decay,eps_eval,object
40,2.294444,0.001,0.99,1,0.9999,0.0000,(8):Q_Learning
58,2.112500,0.001,1.01,1,0.9999,0.0000,(8):Q_Learning
55,1.994444,0.001,1.01,1,0.9900,0.0001,(8):Q_Learning
59,1.936111,0.001,1.01,1,0.9999,0.0001,(8):Q_Learning
38,1.927778,0.001,0.99,1,0.9990,0.0000,(8):Q_Learning
...,...,...,...,...,...,...,...
67,1.331944,0.001,1.01,3,0.9900,0.0001,(8):Q_Learning
35,1.325000,0.010,1.01,3,0.9999,0.0001,(8):Q_Learning
43,1.325000,0.001,0.99,2,0.9900,0.0001,(8):Q_Learning
41,1.286111,0.001,0.99,1,0.9999,0.0001,(8):Q_Learning


In [281]:
sorted_values = [v for _, v in final_scores.items()]

path = foldername + "top_nr_{rank}.pkl"

for i in range(min(len(sorted_values), 3)):
    sorted_values[i]["object"].Save_Q_Table(filename=path.format(rank=i))

In [282]:
strategies_list = Get_Enemy_Strategies()
best_q = sorted_values[0]["object"]
strategies_list.append(best_q)
strategies = Enumerate_Strategies(strategies_list)
gm = Game_Master(environment=prison, strategies=strategies, duel_size=2, max_action_memory=max_action_memory)

d, r = Normal_Tournament(gm=gm, number_of_games=number_of_games_per_epoch, total_games_explicit=total_games_explicit)
r.Sort_Total_Rewards()

normalized_rewards = {k:(v/(number_of_games_per_epoch*len(strategies_list))) for k,v in r.Get_Total_Rewards().items()}

print(normalized_rewards[best_q.Get_ID()])

2.225
